# ============================================================
# NOTEBOOK 04: CLAUDE CROSS-MODEL ROBUSTNESS PROBE
# ============================================================

Generates matched articles with **Claude Haiku 4.5** (`claude-haiku-4-5`) on the
same 80 headlines (prompts A and E), scores them with the same chunked DistilBERT
procedure, and runs a **three-model** cross-model comparison
(GPT-4o-mini + Gemini-2.5-Flash + Claude-Haiku-4.5) against NYT.

**Prereqs:** run 01, 02, 03 first. Needs `experiment_sample.csv`,
`nyt_baseline_axis2.csv`, `axis2_gpt.csv`, `axis2_gemini.csv`.
Requires the `anthropic` SDK (`pip install anthropic`) and `ANTHROPIC_API_KEY`.

**Outputs:** `data/generations/generations_claude.csv`, `data/results/axis2_claude.csv`,
`data/results/results_crossmodel3_summary.csv`, `data/results/results_crossmodel3_paired.csv`.

In [ ]:

# =============================================================================
# CELL 1: Setup & paths
# =============================================================================

import os
import re
import json
import gc
import time
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore", message="Token indices sequence length")
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()  # silence the benign >512-token tokenizer notice
except Exception:
    pass

# Paths - relative to this notebook (notebooks/), same convention as Notebook 01-03
ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed"
GENERATIONS = ROOT / "data" / "generations"
RESULTS = ROOT / "data" / "results"
GENERATIONS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

for f in ["experiment_sample.csv", "nyt_baseline_axis2.csv"]:
    assert (PROCESSED / f).exists(), f"Missing Notebook 01 output: {f}"
assert (RESULTS / "axis2_gpt.csv").exists(), "Missing Notebook 02 output: axis2_gpt.csv"
assert (RESULTS / "axis2_gemini.csv").exists(), "Missing Notebook 03 output: axis2_gemini.csv"

print(f"ROOT:        {ROOT}")
print(f"GENERATIONS: {GENERATIONS}")
print(f"RESULTS:     {RESULTS}")
print("\u2713 All prereq files present.")

In [ ]:

# =============================================================================
# CELL 2: Load Notebook 01/02/03 outputs
# =============================================================================

sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
sampled["pub_date"] = pd.to_datetime(sampled["pub_date"])
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")
gpt_axis2 = pd.read_csv(RESULTS / "axis2_gpt.csv")
gemini_axis2 = pd.read_csv(RESULTS / "axis2_gemini.csv")  # Gemini-2.5-Flash

print(f"Sample:       {len(sampled)} headlines")
print(f"NYT axis2:    {len(nyt_axis2)} rows")
print(f"GPT axis2:    {len(gpt_axis2)} rows")
print(f"Gemini axis2: {len(gemini_axis2)} rows")

In [ ]:

# =============================================================================
# CELL 3: Define prompts A and E (must match Notebook 02/03 exactly)
# =============================================================================

SYSTEM_A = (
    "You are a staff journalist writing for The New York Times. "
    "You write in standard NYT register: clear lede, supporting paragraphs "
    "with context and attribution, sourcing to named or general authorities "
    "(e.g., 'analysts said', 'according to company filings'). "
    "Do not fabricate direct quotes from named individuals you cannot verify. "
    "Write only the article body - no headline, no byline, no editor's note."
)
PROMPT_A = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Begin the article now."""

SYSTEM_E = SYSTEM_A
PROMPT_E = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Use the section-specific frame as the primary organizing logic of the article, while adapting it to the specific headline:
- For Business desk articles, foreground firms, markets, investors, regulation, supply chains, corporate strategy, economic uncertainty, and commercial or technological risk.
- For World desk articles, foreground states, diplomacy, security, governance, rights, sovereignty, international institutions, and geopolitical consequences.

Additional style constraints:
- Avoid promotional, celebratory, or corporate brochure-like language.
- Do not frame the story as a corporate announcement, market opportunity, or institutional success story unless the headline explicitly requires it.
- Lead with concrete stakes, tensions, constraints, risks, or consequences rather than generic background or scene-setting.
- Assume readers already know the basic context about the main entities involved; avoid over-explaining.
- Use sober, direct, plain language.

Begin the article now."""

CLAUDE_PROMPTS = {"A": (SYSTEM_A, PROMPT_A), "E": (SYSTEM_E, PROMPT_E)}
print(f"Defined {len(CLAUDE_PROMPTS)} prompts: {list(CLAUDE_PROMPTS.keys())}")

In [ ]:

# =============================================================================
# CELL 4: API key
# Set your Anthropic key in the environment BEFORE running, e.g. in a terminal:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Do NOT hardcode the key in this notebook - it would leak when the code is shared.
# =============================================================================

assert os.environ.get("ANTHROPIC_API_KEY"), \
    "Set ANTHROPIC_API_KEY in your environment before running this notebook."
print("\u2713 ANTHROPIC_API_KEY is set.")

In [ ]:

# =============================================================================
# CELL 5: Claude API caller (official Anthropic SDK)
# =============================================================================

import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
CLAUDE_MODEL = "claude-haiku-4-5"


def call_claude(system, user, model=CLAUDE_MODEL, temperature=0.7,
                max_tokens=3200, retries=4, sleep_seconds=8):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=max_tokens,
                temperature=temperature,
                system=system,
                messages=[{"role": "user", "content": user}],
            )
            text = "".join(b.text for b in resp.content if b.type == "text").strip()
            if not text:
                raise RuntimeError(f"Empty response (stop_reason={resp.stop_reason})")
            return text
        except Exception as e:
            last_err = e
            print(f"  [retry {attempt}/{retries}] {type(e).__name__}: {e}")
            time.sleep(sleep_seconds * attempt)
    raise RuntimeError(f"Claude failed after {retries} attempts: {last_err}")


# Smoke test
print("Testing Claude connectivity...")
print(call_claude("You are a helpful assistant.", "Reply with exactly the word: pong",
                  temperature=0, max_tokens=20))

In [ ]:

# =============================================================================
# CELL 6: Generate Claude articles (len(sampled) x 2 prompts x REPS) with checkpoint
# =============================================================================

REPS = 3
GEN_FILE = GENERATIONS / "generations_claude.csv"

if GEN_FILE.exists():
    df_claude = pd.read_csv(GEN_FILE)
    done = set(zip(df_claude["sample_id"], df_claude["rep"], df_claude["prompt_version"]))
    rows = df_claude.to_dict("records")
    print(f"Resuming: {len(done)} Claude generations already saved.")
else:
    rows, done = [], set()

target = len(CLAUDE_PROMPTS) * len(sampled) * REPS  # prompts x headlines x reps
print(f"Target: {target}. Done: {len(done)}. To do: {target - len(done)}")

saved_since, SAVE_EVERY = 0, 10
for prompt_name, (sys_msg, user_template) in CLAUDE_PROMPTS.items():
    for _, r in sampled.iterrows():
        sid = r["sample_id"]
        section = "Business" if r["news_desk"] == "Business" else "World"
        date_str = r["pub_date"].strftime("%B %d, %Y").replace(" 0", " ")
        prompt = user_template.format(section=section, date=date_str, headline=r["headline"])
        for rep in range(REPS):
            key = (sid, rep, prompt_name)
            if key in done:
                continue
            print(f"[{prompt_name}] sid={sid} {section[:3]} rep={rep+1}/{REPS}  "
                  f"{r['headline'][:55]}...")
            try:
                text = call_claude(sys_msg, prompt)
                wc = len(text.split())
                print(f"    \u2192 {wc} words")
            except Exception as e:
                print(f"  !! {type(e).__name__}: {e}")
                text, wc = f"[ERROR: {type(e).__name__}]", 0
            rows.append({
                "sample_id": sid, "rep": rep, "prompt_version": prompt_name,
                "headline": r["headline"], "news_desk": r["news_desk"],
                "section_prompted": section, "pub_date": str(r["pub_date"]),
                "generated_full_text": text, "generated_word_count": wc,
                "model": CLAUDE_MODEL,
            })
            saved_since += 1
            if saved_since >= SAVE_EVERY:
                pd.DataFrame(rows).to_csv(GEN_FILE, index=False)
                print(f"    \u2713 checkpoint ({len(rows)} rows)")
                saved_since = 0
            time.sleep(0.3)

df_claude = pd.DataFrame(rows)
df_claude.to_csv(GEN_FILE, index=False)
print(f"\nTotal Claude rows: {len(df_claude)}")
ok = df_claude[~df_claude["generated_full_text"].astype(str).str.startswith("[ERROR")]
print(f"Successful: {len(ok)}")
print("\nWord counts:")
print(ok.groupby(["prompt_version", "news_desk"])["generated_word_count"].describe().round(0))

In [ ]:

# =============================================================================
# CELL 7: Load DistilBERT for sentiment scoring (same config as Notebook 01-03)
# =============================================================================

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS_PER_CHUNK = 450
STRIDE = 50
BATCH_SIZE = 8

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_model.to(device)
sentiment_model.eval()
print(f"Device: {device}")


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def make_token_chunks(text, max_tokens=MAX_TOKENS_PER_CHUNK, stride=STRIDE):
    if not isinstance(text, str) or not text.strip():
        return []
    tids = tokenizer(text, add_special_tokens=False, truncation=False,
                     return_attention_mask=False)["input_ids"]
    if not tids:
        return []
    chunks, start = [], 0
    while start < len(tids):
        end = start + max_tokens
        # version-robust special tokens ([CLS] + content + [SEP]); works on transformers 4.x and 5.x
        chunks.append([tokenizer.cls_token_id] + tids[start:end] + [tokenizer.sep_token_id])
        if end >= len(tids):
            break
        start = end - stride
    return chunks


def predict_token_chunks(token_id_chunks, batch_size=BATCH_SIZE):
    if not token_id_chunks:
        return pd.DataFrame()
    rows_out = []
    for start in range(0, len(token_id_chunks), batch_size):
        batch = token_id_chunks[start:start + batch_size]
        max_len = max(len(x) for x in batch)
        pad_id = tokenizer.pad_token_id
        input_ids, attn = [], []
        for ids in batch:
            pad_n = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad_n)
            attn.append([1] * len(ids) + [0] * pad_n)
        inputs = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long).to(device),
            "attention_mask": torch.tensor(attn, dtype=torch.long).to(device),
        }
        with torch.inference_mode():
            out = sentiment_model(**inputs)
            probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
        for j, prob in enumerate(probs):
            rows_out.append({
                "chunk_id": start + j,
                "neg_prob": float(prob[0]),
                "pos_prob": float(prob[1]),
                "sentiment_score": float(prob[1]) - float(prob[0]),
            })
        clear_memory()
    return pd.DataFrame(rows_out)

In [ ]:

# =============================================================================
# CELL 8: Score Claude generations on the sentiment axis
# =============================================================================

axis2_rows = []
print(f"Scoring {len(df_claude)} Claude generations on the sentiment axis...")
for _, r in tqdm(df_claude.iterrows(), total=len(df_claude)):
    if str(r["generated_full_text"]).startswith("[ERROR"):
        continue
    try:
        chunks = make_token_chunks(r["generated_full_text"])
        chunk_df = predict_token_chunks(chunks)
        if len(chunk_df) > 0:
            axis2_rows.append({
                "sample_id": r["sample_id"], "rep": r["rep"],
                "prompt_version": r["prompt_version"], "news_desk": r["news_desk"],
                "model": "Claude-Haiku-4.5",
                "sentiment": chunk_df["sentiment_score"].mean(),
                "n_chunks": len(chunk_df), "word_count": r["generated_word_count"],
            })
    except Exception as e:
        print(f"Error sid={r['sample_id']} {r['prompt_version']} rep={r['rep']}: {e}")

axis2_claude = pd.DataFrame(axis2_rows)
axis2_claude.to_csv(RESULTS / "axis2_claude.csv", index=False)
print(f"Saved: axis2_claude.csv ({len(axis2_claude)} rows)")

In [ ]:

# =============================================================================
# CELL 9: Three-model cross-model comparison - center and gap (A and E) vs NYT
# =============================================================================

gpt_AE = gpt_axis2[gpt_axis2["prompt_version"].isin(["A", "E"])].copy()
gpt_AE["model"] = "GPT-4o-mini"
gem_AE = gemini_axis2[gemini_axis2["prompt_version"].isin(["A", "E"])].copy()  # model col = Gemini-2.5-Flash

cols = ["sample_id", "rep", "prompt_version", "news_desk", "model", "sentiment"]
all_data = pd.concat([
    nyt_axis2.assign(model="NYT (real)", prompt_version="NYT", rep=-1)[cols],
    gpt_AE[cols],
    gem_AE[cols],
    axis2_claude[cols],
], ignore_index=True)

agg = all_data.groupby(["sample_id", "model", "prompt_version", "news_desk"],
                       as_index=False)["sentiment"].mean()

print("=" * 90)
print("THREE-MODEL CROSS-MODEL COMPARISON - center and gap (n=40/desk)")
print("=" * 90)
print("  Center = (Business mean + Foreign mean) / 2 ; Gap = Business mean - Foreign mean\n")

summary_rows = []
for model_label in ["NYT (real)", "GPT-4o-mini", "Gemini-2.5-Flash", "Claude-Haiku-4.5"]:
    for v in ["NYT", "A", "E"]:
        sub = agg[(agg["model"] == model_label) & (agg["prompt_version"] == v)]
        biz = sub[sub["news_desk"] == "Business"]["sentiment"]
        foreign = sub[sub["news_desk"] == "Foreign"]["sentiment"]
        if not (len(biz) and len(foreign)):
            continue
        center = (biz.mean() + foreign.mean()) / 2
        gap = biz.mean() - foreign.mean()
        print(f"  {model_label:18s} {v:4s}  Biz={biz.mean():+.4f}  For={foreign.mean():+.4f}  "
              f"Gap={gap:+.4f}  Center={center:+.4f}")
        summary_rows.append({"model": model_label, "prompt": v, "n_biz": len(biz),
                             "n_for": len(foreign), "biz_mean": biz.mean(),
                             "for_mean": foreign.mean(), "center": center, "gap": gap})

pd.DataFrame(summary_rows).to_csv(RESULTS / "results_crossmodel3_summary.csv", index=False)
print("\nSaved: results_crossmodel3_summary.csv")

In [ ]:

# =============================================================================
# CELL 10: Per-headline deviation + paired Wilcoxon (3 models vs NYT, within-headline)
# =============================================================================

from scipy.stats import wilcoxon

nyt_per = agg[agg["model"] == "NYT (real)"][["sample_id", "sentiment"]].rename(
    columns={"sentiment": "nyt_sentiment"})
llm = agg[agg["model"].isin(["GPT-4o-mini", "Gemini-2.5-Flash", "Claude-Haiku-4.5"])].merge(
    nyt_per, on="sample_id")
llm["deviation"] = llm["sentiment"] - llm["nyt_sentiment"]

print("Mean per-headline deviation (LLM - NYT) + paired Wilcoxon:\n")
print(f"  {'model':18s} {'pr':2s} {'desk':9s}  n   dev      p")
paired = []
for model_label in ["GPT-4o-mini", "Gemini-2.5-Flash", "Claude-Haiku-4.5"]:
    for v in ["A", "E"]:
        for desk in ["Business", "Foreign"]:
            sub = llm[(llm.model == model_label) & (llm.prompt_version == v)
                      & (llm.news_desk == desk)]
            if len(sub) < 5:
                continue
            try:
                stat, p = wilcoxon(sub["sentiment"].values, sub["nyt_sentiment"].values)
            except Exception:
                stat, p = np.nan, np.nan
            sig = "*" if p < 0.05 else " "
            print(f"  {model_label:18s} {v:2s} {desk:9s} {len(sub):2d}  "
                  f"{sub['deviation'].mean():+.3f}  {p:.4f}{sig}")
            paired.append({"model": model_label, "prompt": v, "desk": desk, "n": len(sub),
                           "deviation": sub["deviation"].mean(),
                           "wilcoxon_stat": stat, "p_value": p})

pd.DataFrame(paired).to_csv(RESULTS / "results_crossmodel3_paired.csv", index=False)
print("\nSaved: results_crossmodel3_paired.csv")

In [ ]:

# =============================================================================
# CELL 11: Final summary
# =============================================================================

print("=" * 78)
print("NOTEBOOK 04 (Claude) COMPLETE")
print("=" * 78)
outs = (sorted(GENERATIONS.glob("generations_claude*"))
        + [RESULTS / "axis2_claude.csv"]
        + sorted(RESULTS.glob("*crossmodel3*")))
for f in outs:
    if f.exists():
        print(f"  {f.relative_to(ROOT)}  {f.stat().st_size:,} bytes")
print("\n\u2192 Notebook 05 (figures) can now include Claude-Haiku-4.5 as a third model.")